# Reddit Bot / Automation-Risk Detection

This notebook is the user-level rule-based automation-risk stage.

It expects the parsed user table produced by `reddit_raw_schema.ipynb`:

`data/interim/reddit/parsed/users.parquet`

The notebook keeps the original scoring logic and produces scored users plus
review samples. A high score is an automation-risk signal, not proof that an
account is a bot.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    """
    Find the repository root from the current working directory.
    Expected repo markers: src/ and README.md.
    """
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "src").exists()
            and (candidate / "README.md").exists()
        ):
            return candidate

    # Fallback: current working directory.
    return cwd


PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
PARSED_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "reddit"
    / "parsed"
)

USERS_FILE = (
    PARSED_OUTPUT_DIR
    / "users.parquet"
)

if not USERS_FILE.exists():
    raise FileNotFoundError(
        f"Missing {USERS_FILE}. "
        "Run reddit_raw_schema.ipynb first."
    )

users_df = pd.read_parquet(
    USERS_FILE
)

print("Loaded users:", users_df.shape)


In [ ]:
# primary regulation
MIN_INTERACTIONS = 5
FAST_TIME_REFERENCE_SECONDS = 3600 

users = users_df.copy()

required_columns = [
    "author",
    "total_interactions",
    "posts_participated",
    "top_level_comments",
    "replies",
    "reply_ratio",
    "unique_parent_authors",
    "unique_threads",
    "max_depth",
    "avg_depth",
    "mean_score",
    "median_score",
    "mean_word_count",
    "median_word_count",
    "mean_lexical_diversity",
    "url_interaction_ratio",
    "total_urls",
    "exact_duplicate_ratio",
    "unique_text_ratio",
    "median_interarrival_seconds",
    "min_interarrival_seconds",
    "median_response_time_seconds",
    "very_fast_reply_ratio_60s",
    "active_utc_hours",
    "hour_coverage_ratio",
    "self_reply_ratio",
    "first_activity_utc",
    "last_activity_utc",
]

missing_columns = [col for col in required_columns if col not in users.columns] 

if missing_columns:
    raise ValueError(f"Missing required columns in users DataFrame: {missing_columns}")

In [ ]:
users["first_activity_utc"] = pd.to_datetime(
    users["first_activity_utc"],
    utc=True,
    errors="coerce",
)

users["last_activity_utc"] = pd.to_datetime(
    users["last_activity_utc"],
    utc=True,
    errors="coerce",
)

numeric_columns = [
    column
    for column in required_columns
    if column not in [
        "author",
        "first_activity_utc",
        "last_activity_utc",
    ]
]

for column in numeric_columns:
    users[column] = pd.to_numeric(
        users[column],
        errors="coerce",
    )


In [ ]:
ratio_columns = [
    "reply_ratio",
    "mean_lexical_diversity",
    "url_interaction_ratio",
    "exact_duplicate_ratio",
    "unique_text_ratio",
    "very_fast_reply_ratio_60s",
    "hour_coverage_ratio",
    "self_reply_ratio",
]

for column in ratio_columns:
    users[column] = users[column].clip(lower=0.0, upper=1.0)

time_columns = [
    "median_interarrival_seconds",
    "min_interarrival_seconds",
    "median_response_time_seconds",
]

for column in time_columns:
    users[column] = users[column].clip(lower=0.0)

In [ ]:
users["activity_span_hours"] = (
    (users["last_activity_utc"] - users["first_activity_utc"]).dt.total_seconds() / 3600
).clip(lower=0.0)

effective_span_hours = users["activity_span_hours"].clip(lower=1)

users["interactions_per_active_hour"] = users["total_interactions"] / effective_span_hours

users["interactions_per_post"] = (
    users["total_interactions"] 
    / users['posts_participated']
    .replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)

users["parent_author_diversity"] = (
    users["unique_parent_authors"] / users["replies"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0.0, upper=1.0)

users["thread_diversity_ratio"] = (
    users["unique_threads"] / users["total_interactions"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0.0, upper=1.0)

users["text_repetition_ratio"] = (1 - users["unique_text_ratio"].fillna(1)).clip(lower=0.0, upper=1.0)

In [ ]:
users["evidence_level"] = pd.cut(
    users["total_interactions"],
    bins=[-1, 2, 4, 9, float('inf')], 
    labels=['insufficient', 'low', 'medium', 'high']
)

users["eligible_for_scoring"] = (
    users["total_interactions"]
    >= MIN_INTERACTIONS
)

users["has_interarrival_evidence"] = (
    users["total_interactions"] >= 2
).astype(int)

users["has_response_evidence"] = (
    users["replies"] >= 1
).astype(int)

In [ ]:
ratio_fill_zero = [
    "exact_duplicate_ratio",
    "very_fast_reply_ratio_60s",
    "url_interaction_ratio",
    "hour_coverage_ratio",
    "self_reply_ratio",
]

users[ratio_fill_zero] = (
    users[ratio_fill_zero]
    .fillna(0)
)

users["unique_text_ratio"] = (
    users["unique_text_ratio"]
    .fillna(1)
)

lexical_median = (
    users["mean_lexical_diversity"]
    .median()
)

users["mean_lexical_diversity"] = (
    users["mean_lexical_diversity"]
    .fillna(lexical_median)
)


users["median_interarrival_seconds_filled"] = (
    users["median_interarrival_seconds"]
    .fillna(FAST_TIME_REFERENCE_SECONDS)
)

users["median_response_time_seconds_filled"] = (
    users["median_response_time_seconds"]
    .fillna(FAST_TIME_REFERENCE_SECONDS)
)

In [ ]:
users["duplicate_score"] = (0.70 * users["exact_duplicate_ratio"].clip(0, 1) +
    0.30 * ( 1 - users["unique_text_ratio"].clip(0, 1))).clip(0, 1)

response_time_component = (
    1
    - (
        users["median_response_time_seconds_filled"]
        .clip(
            lower=0,
            upper=FAST_TIME_REFERENCE_SECONDS,
        )
        / FAST_TIME_REFERENCE_SECONDS
    )
).clip(0, 1)

users["fast_reply_score"] = ( 0.70 * users["very_fast_reply_ratio_60s"].clip(0, 1)
    +
    0.30 * response_time_component
).clip(0, 1)

users.loc[
    users["replies"].fillna(0) == 0,
    "fast_reply_score",] = 0

In [ ]:
users["rapid_activity_score"] = (
    1
    - (
        users["median_interarrival_seconds_filled"]
        .clip(
            lower=0,
            upper=FAST_TIME_REFERENCE_SECONDS,
        )
        / FAST_TIME_REFERENCE_SECONDS
    )
).clip(0, 1)

users.loc[
    users["total_interactions"].fillna(0) <= 1,
    "rapid_activity_score",
] = 0

In [ ]:
activity_reliability = (
    users["total_interactions"] / 20
).clip(0, 1)


users["continuous_activity_score"] = (
    users["hour_coverage_ratio"].clip(0, 1)
    * activity_reliability
).clip(0, 1)

In [ ]:
users["url_score"] = (
    users["url_interaction_ratio"]
    .clip(0, 1)
)

lexical_q10 = (
    users["mean_lexical_diversity"]
    .quantile(0.10)
)

lexical_q50 = (
    users["mean_lexical_diversity"]
    .quantile(0.50)
)

lexical_range = max(
    lexical_q50 - lexical_q10,
    1e-9,
)


users["low_diversity_score"] = (
    (
        lexical_q50
        - users["mean_lexical_diversity"]
    )
    / lexical_range
).clip(0, 1).fillna(0)

In [ ]:
users["self_reply_score"] = (
    users["self_reply_ratio"]
    .clip(0, 1)
)

users["multi_post_activity_score"] = (
    users["posts_participated"]
    .fillna(0)
    .rank(
        method="average",
        pct=True,
    )
    .clip(0, 1)
)

users["rule_automation_score_v1"] = (
    25 * users["duplicate_score"]
    + 15 * users["fast_reply_score"]
    + 15 * users["rapid_activity_score"]
    + 10 * users["continuous_activity_score"]
    + 10 * users["url_score"]
    + 10 * users["low_diversity_score"]
    + 5 * users["self_reply_score"]
    + 10 * users["multi_post_activity_score"]
).clip(0, 100)

In [ ]:
users["rule_risk_group"] = pd.cut(
    users["rule_automation_score_v1"],
    bins=[
        -0.001,
        20,
        50,
        75,
        100,
    ],
    labels=[
        "low_risk",
        "review_low",
        "review_high",
        "high_risk",
    ],
    include_lowest=True,
)

users["score_is_reliable"] = (
    users["eligible_for_scoring"]
)


users["final_rule_status"] = (
    users["rule_risk_group"]
    .astype("string")
)


users.loc[
    ~users["eligible_for_scoring"],
    "final_rule_status",
] = "insufficient_evidence"

In [ ]:
score_summary = (
    users.loc[
        users["eligible_for_scoring"],
        "rule_automation_score_v1",
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


status_counts = (
    users["final_rule_status"]
    .value_counts(
        dropna=False,
    )
    .rename_axis("status")
    .to_frame("number_of_users")
)


review_columns = [
    "author",
    "author_fullname",
    "total_interactions",
    "posts_participated",
    "replies",
    "evidence_level",
    "rule_automation_score_v1",
    "final_rule_status",
    "duplicate_score",
    "fast_reply_score",
    "rapid_activity_score",
    "continuous_activity_score",
    "url_score",
    "low_diversity_score",
    "self_reply_score",
    "multi_post_activity_score",
    "exact_duplicate_ratio",
    "unique_text_ratio",
    "median_interarrival_seconds",
    "median_response_time_seconds",
    "very_fast_reply_ratio_60s",
    "hour_coverage_ratio",
    "url_interaction_ratio",
    "mean_lexical_diversity",
]


top_suspicious = (
    users.loc[
        users["eligible_for_scoring"],
        review_columns,
    ]
    .sort_values(
        [
            "rule_automation_score_v1",
            "total_interactions",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(30)
    .reset_index(drop=True)
)


lowest_risk = (
    users.loc[
        users["eligible_for_scoring"],
        review_columns,
    ]
    .sort_values(
        [
            "rule_automation_score_v1",
            "total_interactions",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .head(30)
    .reset_index(drop=True)
)


eligible_scores = users.loc[
    users["eligible_for_scoring"],
    "rule_automation_score_v1",
]

if not eligible_scores.empty:
    score_median = eligible_scores.median()

    middle_sample = (
        users.loc[
            users["eligible_for_scoring"],
            review_columns,
        ]
        .assign(
            distance_from_median=lambda df: (
                df["rule_automation_score_v1"]
                - score_median
            ).abs()
        )
        .sort_values("distance_from_median")
        .head(30)
        .drop(columns="distance_from_median")
        .reset_index(drop=True)
    )
else:
    middle_sample = pd.DataFrame(
        columns=review_columns
    )



users_scored_df = users.copy()

In [ ]:

print("Total number of users:", len(users_scored_df))

print(
    "Number of users with sufficient evidence:",
    users_scored_df["eligible_for_scoring"].sum(),
)

print("\nScore summary:")
print(score_summary)

print("\nNumber of users in each group:")
print(status_counts)



In [ ]:
top_suspicious

In [ ]:
BOT_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "reddit"
    / "bot_detection"
)

BOT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

users_scored_df.to_parquet(
    BOT_OUTPUT_DIR / "users_scored.parquet",
    index=False,
)

users_scored_df.to_csv(
    BOT_OUTPUT_DIR / "users_scored.csv",
    index=False,
)

top_suspicious.to_csv(
    BOT_OUTPUT_DIR / "top_suspicious_review_sample.csv",
    index=False,
)

lowest_risk.to_csv(
    BOT_OUTPUT_DIR / "lowest_risk_review_sample.csv",
    index=False,
)

middle_sample.to_csv(
    BOT_OUTPUT_DIR / "middle_score_review_sample.csv",
    index=False,
)

status_counts.to_csv(
    BOT_OUTPUT_DIR / "status_counts.csv",
)

print("Saved bot-detection outputs to:", BOT_OUTPUT_DIR)
